# Tesla Stock — Pipeline Big Data & Machine Learning
## Projet Écosystème Hadoop — ESGI 2025-2026

Pipeline complète : HDFS → Spark (parsing, features) → MLlib (modèles) → Data Viz.

**Question centrale du projet** : peut-on prédire le cours de Tesla, et à quel horizon ?
Réponse construite en comparant systématiquement :
- 3 horizons de prédiction : **J+1 (24h), J+2 (48h), J+5 (1 semaine)**
- 2 périmètres de données : **historique complet (2010-2024)** vs **depuis 2020 uniquement**
- 3 modèles MLlib + baselines de référence

Contraintes techniques respectées : tout le calcul est en PySpark (API DataFrame + SQL), pandas n'est utilisé que pour l'affichage via `.toPandas()`.

## 1. Session Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

spark = (SparkSession.builder
    .appName("Tesla - Pipeline ML")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 3.3.0


## 2. Chargement depuis HDFS

Schéma **explicite** plutôt qu'`inferSchema` : plus rapide (une seule lecture du fichier) et aucun risque que Spark devine un mauvais type. La vérification de la période se fait avec l'API de base : `select` sur `min`/`max` de la colonne Date.

In [2]:
schema = StructType([
    StructField("Date", StringType(), True),
    StructField("Open", DoubleType(), True),
    StructField("High", DoubleType(), True),
    StructField("Low", DoubleType(), True),
    StructField("Close", DoubleType(), True),
    StructField("Adj Close", DoubleType(), True),
    StructField("Volume", LongType(), True),
])

df = (spark.read.option("header", True).schema(schema)
    .csv("hdfs://namenode:9000/data/tesla/raw/tesla.csv")
    .withColumn("Date", F.to_date("Date"))
    .dropDuplicates(["Date"]))

df.select(F.count("*").alias("lignes"), F.min("Date").alias("debut"), F.max("Date").alias("fin")).show()

+------+----------+----------+
|lignes|     debut|       fin|
+------+----------+----------+
|  3562|2010-06-29|2024-08-22|
+------+----------+----------+



## 3. Statistiques descriptives en Spark

Agrégations calculées directement avec l'API DataFrame de Spark (`groupBy` / `agg`). Tout le calcul est distribué par Spark, aucun passage par pandas.

In [ ]:
(df.groupBy(F.year("Date").alias("annee"))
   .agg(
       F.round(F.avg("Close"), 2).alias("prix_moyen"),
       F.round(F.min("Close"), 2).alias("prix_min"),
       F.round(F.max("Close"), 2).alias("prix_max"),
       F.round(F.avg("Volume") / 1e6, 1).alias("volume_moyen_M"))
   .orderBy("annee")
   .show(20))

+-----+----------+--------+--------+--------------+
|annee|prix_moyen|prix_min|prix_max|volume_moyen_M|
+-----+----------+--------+--------+--------------+
| 2010|      1.56|    1.05|    2.36|          23.7|
| 2011|      1.79|    1.46|    2.33|          19.4|
| 2012|      2.08|    1.52|    2.53|          18.4|
| 2013|      6.96|    2.19|   12.89|         126.1|
| 2014|     14.89|    9.29|   19.07|         103.7|
| 2015|     15.34|   12.33|   18.82|          64.8|
| 2016|     13.98|    9.58|   17.69|          69.2|
| 2017|     20.95|   14.47|   25.67|          95.0|
| 2018|     21.15|    16.7|    25.3|         129.2|
| 2019|     18.24|   11.93|   28.73|         137.4|
| 2020|     96.67|   24.08|  235.22|         225.9|
| 2021|     260.0|  187.67|  409.97|          82.2|
| 2022|    263.09|   109.1|  399.93|          86.9|
| 2023|    217.48|   108.1|  293.34|         137.3|
| 2024|    194.76|  142.05|  263.26|          98.7|
+-----+----------+--------+--------+--------------+



## 4. Détection des jours aberrants (outliers)

Un outlier est un jour dont le rendement s'écarte anormalement du comportement habituel. Détection par **z-score** : nombre d'écarts-types entre le rendement du jour et le rendement moyen. Seuil classique `|z| > 3`.

Le filtrage se fait directement avec `where`, sans DataFrame intermédiaire. Ces jours extrêmes sont de vrais événements de marché (résultats, annonces), pas des erreurs de données : on les documente, on ne les supprime pas (les tests menés en parallèle montrent que leur retrait ne change quasiment rien aux performances : R² moyen 0.563 avec vs 0.559 sans).

In [4]:
w = Window.orderBy("Date")
df_ret = df.withColumn("Return", (F.col("Close") - F.lag("Close").over(w)) / F.lag("Close").over(w))

stats = df_ret.select(F.mean("Return").alias("m"), F.stddev("Return").alias("s")).first()
mean_ret, std_ret = stats["m"], stats["s"]
print(f"Rendement moyen quotidien : {mean_ret:.5f}")
print(f"Ecart-type quotidien : {std_ret:.5f}")

df_ret = df_ret.withColumn("z_score", (F.col("Return") - F.lit(mean_ret)) / F.lit(std_ret))

print("Jours aberrants (|z| > 3) :", df_ret.where(F.abs(F.col("z_score")) > 3).count())
(df_ret.where(F.abs(F.col("z_score")) > 3)
    .select("Date", "Close", "Return", "z_score")
    .orderBy(F.abs(F.col("z_score")).desc())
    .show(10))

Rendement moyen quotidien : 0.00201
Ecart-type quotidien : 0.03589
Jours aberrants (|z| > 3) : 61
+----------+----------+--------------------+------------------+
|      Date|     Close|              Return|           z_score|
+----------+----------+--------------------+------------------+
|2013-05-09|  4.626667| 0.24395072987549127| 6.741845896924955|
|2020-09-08|    110.07|-0.21062823851651982|-5.925480199025753|
|2020-02-03|      52.0| 0.19894859586288038| 5.487813602892772|
|2012-01-13|  1.519333|-0.19327437049103896|-5.441896222225295|
|2021-03-09|224.526672|  0.1964120725708824|5.4171306854595915|
|2010-11-10|  1.957333| 0.19204202192448241|   5.2953545795619|
|2020-03-16| 29.671333|-0.18577807826683834|-5.233004072088246|
|2020-03-19| 28.509333| 0.18387686429152408| 5.067823783917326|
|2019-10-24| 19.978666| 0.17669232671165097|4.8676190082877095|
|2020-02-05|     48.98| -0.1717583882249137|-4.842330508475281|
+----------+----------+--------------------+------------------+
only s

## 5. Le piège méthodologique : pourquoi prédire le prix brut est une illusion

**Point clé de la soutenance.** Une première version du modèle prédisait le prix de clôture du lendemain et obtenait un R² de 0.97. Ce score paraît excellent, mais il ne mesure aucune capacité de prédiction.

Démonstration : la "prédiction" la plus bête possible, *le prix de demain = le prix d'aujourd'hui*, sans le moindre machine learning. Si cette règle triviale obtient déjà un R² proche de 1, alors le R² sur le prix brut ne prouve rien : le prix d'un jour est simplement très proche du prix de la veille (comportement de **marche aléatoire**).

In [5]:
naive = (df.withColumn("prediction", F.lag("Close").over(w))
    .withColumn("Target", F.col("Close"))
    .dropna())

mean_target = naive.select(F.mean("Target")).first()[0]

naive.select(
    F.round(F.mean(F.abs(F.col("Target") - F.col("prediction"))), 3).alias("MAE"),
    F.round(F.sqrt(F.mean(F.pow(F.col("Target") - F.col("prediction"), 2))), 3).alias("RMSE"),
    F.round(F.lit(1) - F.sum(F.pow(F.col("Target") - F.col("prediction"), 2)) /
            F.sum(F.pow(F.col("Target") - F.lit(mean_target), 2)), 4).alias("R2"),
).show()

+-----+-----+------+
|  MAE| RMSE|    R2|
+-----+-----+------+
|2.093|4.814|0.9978|
+-----+-----+------+



Un R² quasi parfait obtenu sans aucun modèle : c'est la preuve que **le R² sur le prix brut est un artefact**, pas un signal.

**Conséquence méthodologique** : la cible correcte est le **rendement** (la variation en %), pas le prix. Prédire le rendement est beaucoup plus difficile, mais c'est la seule mesure honnête. Et la vraie métrique métier devient l'**accuracy directionnelle** : à quelle fréquence le modèle devine-t-il correctement si ça monte ou si ça baisse ?

## 6. Feature engineering

Toutes les features sont **stationnaires** : rendements, ratios prix/moyenne mobile, volatilité, ratio de volume. Jamais de prix bruts en entrée, car un modèle entraîné sur Tesla à 20$ ne saurait rien faire de Tesla à 400$.

La cible `Target_return` est le rendement à `horizon` jours : `(Close_futur - Close) / Close`. Le paramètre `horizon` permet de tester 24h, 48h et 1 semaine avec la même fonction.

In [6]:
def build_features(sdf, horizon=1):
    w_ord = Window.orderBy("Date")
    w5 = Window.orderBy("Date").rowsBetween(-4, 0)
    w10 = Window.orderBy("Date").rowsBetween(-9, 0)
    w20 = Window.orderBy("Date").rowsBetween(-19, 0)

    return (sdf
        .withColumn("Return", (F.col("Close") - F.lag("Close").over(w_ord)) / F.lag("Close").over(w_ord))
        .withColumn("Return_lag1", F.lag("Return", 1).over(w_ord))
        .withColumn("Return_lag2", F.lag("Return", 2).over(w_ord))
        .withColumn("Return_lag5", F.lag("Return", 5).over(w_ord))
        .withColumn("Volatility10", F.stddev("Return").over(w10))
        .withColumn("Close_to_MA5", F.col("Close") / F.avg("Close").over(w5))
        .withColumn("Close_to_MA10", F.col("Close") / F.avg("Close").over(w10))
        .withColumn("Close_to_MA20", F.col("Close") / F.avg("Close").over(w20))
        .withColumn("Range_pct", (F.col("High") - F.col("Low")) / F.col("Close"))
        .withColumn("Volume_ratio", F.col("Volume") / F.avg("Volume").over(w20))
        .withColumn("Target_return", (F.lead("Close", horizon).over(w_ord) - F.col("Close")) / F.col("Close"))
        .dropna())

FEATURES = ["Return", "Return_lag1", "Return_lag2", "Return_lag5",
            "Volatility10", "Close_to_MA5", "Close_to_MA10", "Close_to_MA20",
            "Range_pct", "Volume_ratio"]
print("Features :", FEATURES)

Features : ['Return', 'Return_lag1', 'Return_lag2', 'Return_lag5', 'Volatility10', 'Close_to_MA5', 'Close_to_MA10', 'Close_to_MA20', 'Range_pct', 'Volume_ratio']


## 7. Protocole d'expérimentation

Trois garde-fous méthodologiques :

1. **Split chronologique** : la date de coupure est obtenue par `select max(Date)` sur les 80% premières lignes, puis train et test sont séparés par un `where` sur les dates. Jamais de split aléatoire sur une série temporelle (le modèle verrait le futur).
2. **Scaler calé sur le train uniquement** : la normalisation apprend la moyenne et l'écart-type sur les données d'entraînement seules, puis s'applique au test. Caler sur tout le dataset ferait fuiter de l'information du test vers l'entraînement.
3. **Baseline systématique** : chaque configuration est comparée à la stratégie "toujours prédire la classe majoritaire". Un modèle ne vaut que s'il bat cette référence.

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

def chrono_split(sdf, ratio=0.8):
    total = sdf.count()
    cutoff = (sdf.orderBy("Date").limit(int(total * ratio))
              .select(F.max("Date")).first()[0])
    train = sdf.where(F.col("Date") <= F.lit(cutoff))
    test = sdf.where(F.col("Date") > F.lit(cutoff))
    return train, test, cutoff

def run_experiment(sdf, label):
    assembler = VectorAssembler(inputCols=FEATURES, outputCol="features_raw")
    assembled = assembler.transform(sdf)

    train_raw, test_raw, cutoff = chrono_split(assembled)

    scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)
    scaler_model = scaler.fit(train_raw)
    train = scaler_model.transform(train_raw)
    test = scaler_model.transform(test_raw)

    models = {
        "Linear Regression": LinearRegression(featuresCol="features", labelCol="Target_return"),
        "Random Forest": RandomForestRegressor(featuresCol="features", labelCol="Target_return", numTrees=100, maxDepth=8, seed=42),
        "GBT": GBTRegressor(featuresCol="features", labelCol="Target_return", maxIter=60, maxDepth=5, seed=42),
    }
    ev = {m: RegressionEvaluator(labelCol="Target_return", predictionCol="prediction", metricName=m)
          for m in ["mae", "rmse", "r2"]}

    rows, preds_store = [], {}
    for name, model in models.items():
        fitted = model.fit(train)
        preds = fitted.transform(test)
        mae = ev["mae"].evaluate(preds)
        rmse = ev["rmse"].evaluate(preds)
        r2 = ev["r2"].evaluate(preds)
        acc = preds.where((F.col("prediction") > 0) == (F.col("Target_return") > 0)).count() / preds.count()
        rows.append((label, name, float(mae), float(rmse), float(r2), float(acc)))
        preds_store[name] = preds
        print(f"[{label}] {name}: MAE={mae:.5f} RMSE={rmse:.5f} R2={r2:.4f} DirAcc={acc:.3f}")

    up = test.where(F.col("Target_return") > 0).count() / test.count()
    base = max(up, 1 - up)
    rows.append((label, "Baseline majoritaire", None, None, None, float(base)))
    print(f"[{label}] Baseline majoritaire : {base:.3f}")
    return rows, preds_store

## 8. Grille d'expériences : horizon × historique

La question posée en soutenance : *quelle est la meilleure façon d'utiliser notre modèle ?* Sous 24h, 48h ou 1 semaine, et en gardant ou non l'historique d'avant 2020 (période où Tesla était une smallcap très spéculative, potentiellement non représentative du marché actuel).

Le périmètre "depuis 2020" est un simple `where` sur la date, pas un dataset séparé.

In [ ]:
datasets = {
    "Historique complet": df,
    "Depuis 2020": df.where(F.col("Date") >= F.lit("2020-01-01")),
}
horizons = {"J+1 (24h)": 1, "J+2 (48h)": 2, "J+5 (1 semaine)": 5}

all_rows, all_preds = [], {}
for ds_name, ds in datasets.items():
    for h_name, h in horizons.items():
        label = f"{ds_name} | {h_name}"
        feat = build_features(ds, horizon=h)
        rows, preds = run_experiment(feat, label)
        all_rows.extend(rows)
        all_preds[label] = preds

print("\nToutes les configurations sont entrainees.")

## 9. Résultats et choix du meilleur modèle

Les résultats sont chargés dans un DataFrame Spark et manipulés avec l'API DataFrame. Le meilleur modèle est sélectionné par configuration via une window function (`row_number` partitionné par expérience), le tout en PySpark.

In [ ]:
res_schema = StructType([
    StructField("Experiment", StringType()),
    StructField("Model", StringType()),
    StructField("MAE", DoubleType()),
    StructField("RMSE", DoubleType()),
    StructField("R2", DoubleType()),
    StructField("DirAcc", DoubleType()),
])
results = spark.createDataFrame(all_rows, res_schema)

(results
    .where(F.col("Model") != "Baseline majoritaire")
    .select(
        "Experiment",
        "Model",
        F.round("MAE", 5).alias("MAE"),
        F.round("R2", 4).alias("R2"),
        F.round("DirAcc", 3).alias("DirAcc"))
    .orderBy(F.col("DirAcc").desc())
    .show(18, truncate=False))

In [ ]:
w_exp = Window.partitionBy("Experiment").orderBy(F.col("DirAcc").desc())

best = (results.where(F.col("Model") != "Baseline majoritaire")
    .withColumn("rang", F.row_number().over(w_exp))
    .where(F.col("rang") == 1)
    .join(results.where(F.col("Model") == "Baseline majoritaire")
          .select("Experiment", F.col("DirAcc").alias("Baseline")), on="Experiment")
    .select("Experiment", "Model",
            F.round("MAE", 5).alias("MAE"),
            F.round("DirAcc", 3).alias("DirAcc"),
            F.round("Baseline", 3).alias("Baseline"),
            F.round(F.col("DirAcc") - F.col("Baseline"), 3).alias("Gain_vs_baseline")))

best.orderBy(F.col("Gain_vs_baseline").desc()).show(truncate=False)

## 10. Data Viz

pandas et matplotlib n'interviennent qu'ici, uniquement pour l'affichage via `.toPandas()`. Tout le calcul a été fait en Spark en amont.

**Graphique 1 — L'illusion du prix brut.** À gauche, le prix réel et la "prédiction" naïve (prix de la veille) : les courbes sont indiscernables, d'où le R² de 0.99 sans aucun modèle. À droite, le rendement d'aujourd'hui face à celui de demain : un nuage sans structure, corrélation quasi nulle. C'est la même donnée vue des deux côtés : niveau prévisible, variation imprévisible.

In [ ]:
import matplotlib.pyplot as plt

pdf_price = (naive.where(F.col("Date") >= F.lit("2024-01-01"))
    .select("Date", "Target", "prediction").orderBy("Date").toPandas())

pdf_scatter = (df_ret
    .withColumn("Return_demain", F.lead("Return").over(w))
    .select("Return", "Return_demain").dropna().toPandas())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(pdf_price["Date"], pdf_price["Target"], label="Prix reel", linewidth=1.5)
axes[0].plot(pdf_price["Date"], pdf_price["prediction"], label="Prediction naive (prix de la veille)", linewidth=1, linestyle="--")
axes[0].set_title("Le prix 'se predit' tout seul (R2 = 0.99 sans ML)")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=45)

axes[1].scatter(pdf_scatter["Return"], pdf_scatter["Return_demain"], s=4, alpha=0.3)
axes[1].axhline(0, color="grey", linewidth=0.5)
axes[1].axvline(0, color="grey", linewidth=0.5)
corr = pdf_scatter["Return"].corr(pdf_scatter["Return_demain"])
axes[1].set_title(f"Rendement aujourd'hui vs demain (corr = {corr:.4f})")
axes[1].set_xlabel("Rendement jour J")
axes[1].set_ylabel("Rendement jour J+1")

plt.tight_layout()
plt.show()

**Graphique 2 — Accuracy directionnelle par configuration.** Chaque barre est le meilleur modèle de sa configuration, la ligne noire est sa baseline majoritaire. L'écart entre les deux est le seul vrai apport du machine learning.

In [ ]:
pdf_res = results.toPandas()

models_pdf = pdf_res[pdf_res["Model"] != "Baseline majoritaire"]
base_pdf = pdf_res[pdf_res["Model"] == "Baseline majoritaire"][["Experiment", "DirAcc"]]
best_pdf = models_pdf.loc[models_pdf.groupby("Experiment")["DirAcc"].idxmax()].merge(
    base_pdf, on="Experiment", suffixes=("", "_baseline"))

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(best_pdf))
ax.bar(x, best_pdf["DirAcc"], width=0.55, label="Meilleur modele")
for i, b in zip(x, best_pdf["DirAcc_baseline"]):
    ax.hlines(b, i - 0.35, i + 0.35, color="black", linewidth=2)
ax.axhline(0.5, color="red", linestyle=":", linewidth=1, label="Hasard (50%)")
ax.set_xticks(list(x))
ax.set_xticklabels([e.replace(" | ", "\n") for e in best_pdf["Experiment"]], fontsize=8)
ax.set_ylim(0.4, 0.7)
ax.set_ylabel("Accuracy directionnelle")
ax.set_title("Predire la direction : meilleur modele vs baseline majoritaire (trait noir)")
ax.legend()
plt.tight_layout()
plt.show()

**Graphique 3 — L'erreur croît avec l'horizon.** La MAE du rendement prédit augmente mécaniquement de J+1 à J+5 : l'incertitude s'accumule avec le temps. Sur le prix brut, cette dégradation se lisait dans la chute du R² (0.97 → 0.87 → 0.41) : c'est la signature de la marche aléatoire, pas la preuve d'un signal court terme.

In [ ]:
piv = (models_pdf.assign(
        Horizon=models_pdf["Experiment"].str.split(" \\| ").str[1],
        Dataset=models_pdf["Experiment"].str.split(" \\| ").str[0])
    .pivot_table(index="Horizon", columns="Dataset", values="MAE", aggfunc="mean")
    .reindex(["J+1 (24h)", "J+2 (48h)", "J+5 (1 semaine)"]))

ax = piv.plot(kind="bar", figsize=(9, 4.5), rot=0)
ax.set_ylabel("MAE moyenne du rendement predit")
ax.set_title("L'erreur de prediction augmente avec l'horizon")
plt.tight_layout()
plt.show()

## 11. Synthèse — ce que les chiffres nous apprennent vraiment

**1. Le R² de 0.97 sur le prix brut était une illusion.** Une règle triviale sans ML (prix de demain = prix d'aujourd'hui) atteint 0.99. Le cours de Tesla se comporte comme une marche aléatoire : le *niveau* du prix est trivialement prévisible, sa *variation* ne l'est presque pas (corrélation jour J / jour J+1 ≈ 0).

**2. Sur la cible honnête (le rendement), le marché est très difficile à battre.** R² proche de 0 pour tous les modèles, accuracy directionnelle de 50 à 58% contre une baseline à 52-58%. Le gain du ML se compte en points de pourcentage, pas en dizaines. C'est cohérent avec la théorie des marchés efficients, et c'est le résultat *attendu* d'une démarche propre.

**3. Meilleure façon d'utiliser le modèle** (réponse à la question de cadrage) : l'horizon **court (24-48h)** minimise l'erreur absolue, l'erreur croissant mécaniquement avec l'horizon. Le choix de l'historique (complet vs depuis 2020) se lit dans le tableau de la section 9 : comparer le gain vs baseline de chaque périmètre, et retenir celui où le modèle bat le plus nettement sa référence.

**4. La vraie valeur du projet est méthodologique** : split chronologique strict, scaler calé sur le train, features stationnaires, baselines systématiques, et la capacité à reconnaître un score trompeur. Un modèle qui affiche honnêtement 55% en battant sa baseline vaut mieux qu'un modèle qui affiche 97% en ne prédisant rien.